### Definindo estruturas

In [2]:
(define mapas '())

(define mapa-schema
  '((tipo . 0) (nome . 1) (largura . 2) (altura . 3) (objetos . 4)))

(define objeto-schema
  '((tipo . 0) (nome . 1) (posicao . 2)))

### Primitivas

In [3]:
;primitiva para acessar campos de um registro
(define (acessa item record)
  (cond ((eq? (car record) 'mapa) (list-ref record (cdr (assq item mapa-schema))))
        ((eq? (car record) 'objeto) (list-ref record (cdr (assq item objeto-schema))))
        (else (display "Registro desconhecido\n"))))


;primitiva para verificar se a entidade é a que estamos procurando
(define (igual? entidade nome-entidade) (eq? (acessa 'nome entidade) nome-entidade))


;primitiva para criar mapa:
(define (cria-mapa nome largura altura)
  (let ([novo-mapa (list 'mapa nome largura altura '())]) ;lista vazia para objetos
    (set! mapas (cons novo-mapa mapas))))


(define (localiza-mapa nome-mapa)
  (let ([mapa (car mapas)])
    (do [(resto mapas (cdr resto))
         (m mapa (car resto))]
        [(or (igual? m nome-mapa) (null? m))
          (set! mapa m)])
    (if (null? mapa) (display "Mapa não encontrado\n")
      mapa)))

(define (localiza-objeto nome-objeto nome-mapa)
  (let* ([mapa (localiza 'mapa nome-mapa)]
         [objeto (car (acessa 'objetos mapa))])
    (do [(resto (acessa 'objetos mapa) (cdr resto))
          (obj objeto (car resto))]
        [(or (igual? obj nome-objeto) (null? obj))
          (set! objeto obj)])
    (if (null? objeto) (display "Objeto não encontrado\n")
      objeto)))


;primitiva para localizar entidade
(define (localiza tipo-entidade id . nome-mapa)
  (cond [(eq? tipo-entidade 'mapa) (localiza-mapa id)]
        [(eq? tipo-entidade 'objeto) (localiza-objeto id (car nome-mapa))]
        [else (display "Entidade desconhecida\n")]))

(define (altera-mapa atributo nome-mapa novo-valor)
  (set! mapas
    (map (lambda (mapa)
      (if (igual? mapa nome-mapa)
          (list 'mapa
                (if (eq? atributo 'nome)
                  novo-valor
                  (acessa 'nome mapa))
                (if (eq? atributo 'largura)
                  novo-valor
                  (acessa 'largura mapa))
                (if (eq? atributo 'altura)
                  novo-valor
                  (acessa 'altura mapa))
                (if (eq? atributo 'objetos)
                  novo-valor
                  (acessa 'objetos mapa)))
          mapa)) mapas)))

(define (altera-objeto atributo nome-objeto novo-valor nome-mapa)
  (set! mapas
    (map (lambda (mapa)
      (if (igual? mapa nome-mapa)
        (let* ([objetos (acessa 'objetos mapa)]
               [novo-objetos (map (lambda (objeto)
                  (if (igual? objeto nome-objeto)
                    (list 'objeto
                        (if (eq? atributo 'nome)
                          novo-valor
                          (acessa 'nome objeto))
                        (if (eq? atributo 'posicao)
                          novo-valor
                          (acessa 'posicao objeto)))
                    objeto))
                  objetos)])
          (list 'mapa
                (acessa 'nome mapa)
                (acessa 'largura mapa)
                (acessa 'altura mapa)
                novo-objetos))
        mapa))
      mapas)))


;primitiva para mudar atributo de uma entidade
(define (altera atributo entidade novo-valor . nome-mapa)
  (let ([tipo-entidade (acessa 'tipo entidade)]
        [nome-entidade (acessa 'nome entidade)])
    (cond [(eq? tipo-entidade 'mapa) (altera-mapa atributo nome-entidade novo-valor)]
          [(eq? tipo-entidade 'objeto) (altera-objeto atributo nome-entidade novo-valor (car nome-mapa))]
          [else (display "Entidade desconhecida\n")])))


;primitiva para adicionar objetos a um mapa
(define (adiciona-objeto nome-mapa nome posicao)
  (let* ([novo-objeto (list 'objeto nome posicao)]
         [mapa (localiza 'mapa nome-mapa)]
         [objetos (acessa 'objetos mapa)]
         [novo-objetos (cons novo-objeto objetos)])
    (altera 'objetos mapa novo-objetos)))

### Macros para sintaxe

In [4]:
;; 1. Criação de Mapas
;; Ex: (cria mapa 'mapa-a com largura 1000 e altura 1000)
(define-syntax cria
  (syntax-rules (mapa com largura e altura)
    [(_ mapa nome com largura l e altura a)
     (cria-mapa nome l a)]))

;; 2. Adição de Objetos
;; Ex: (adiciona 'obj-1 ao 'mapa-a em '(100 100))
(define-syntax adiciona
  (syntax-rules (ao em)
    [(_ obj ao mapa em pos)
     (adiciona-objeto mapa obj pos)]))

;; 3. Localização (Busca)
;; Ex: (encontra mapa 'mapa-a)
;; Ex: (encontra objeto 'obj-1 no mapa 'mapa-a)
(define-syntax encontra
  (syntax-rules (mapa objeto no)
    [(_ mapa m)
     (localiza 'mapa m)]
    [(_ objeto obj no mapa m)
     (localiza 'objeto obj m)]))

;; 4. Modificação de Atributos
;; Ex: (muda 'nome do mapa 'mapa-a para 'mapa-novo)
;; Ex: (muda 'nome do objeto 'obj-1 no mapa 'mapa-a para 'obj-modificado)
(define-syntax muda
  (syntax-rules (do mapa para objeto no)
    ;; Padrão para alterar mapas
    [(_ atributo do mapa m para valor)
     (altera atributo (localiza 'mapa m) valor)]

    ;; Padrão para alterar objetos dentro de mapas
    [(_ atributo do objeto obj no mapa m para valor)
     (altera atributo (localiza 'objeto obj m) valor m)]))

### Testes da linguagem

In [5]:
(display "Criando mapas…\n")
(cria mapa 'mapa-principal com largura 1000 e altura 1000)
(cria mapa 'mapa-secundario com largura 2000 e altura 1200)

(display mapas)
(newline)

Criando mapas…
((mapa mapa-secundario 2000 1200 ()) (mapa mapa-principal 1000 1000 ()))


In [6]:
(display "Adicionando objetos ao mapa-principal")
(adiciona 'castelo ao 'mapa-principal em '(100 100))
(adiciona 'torre-mago ao 'mapa-principal em '(100 80))
(adiciona 'estabulo-vazio ao 'mapa-principal em '(80 80))

(newline)
(display mapas)
(newline)

Adicionando objetos ao mapa-principal
((mapa mapa-secundario 2000 1200 ()) (mapa mapa-principal 1000 1000 ((objeto estabulo-vazio (80 80)) (objeto torre-mago (100 80)) (objeto castelo (100 100)))))


In [7]:
(display "Localizando o mapa-principal\n")
(display (encontra mapa 'mapa-principal))
(newline)

Localizando o mapa-principal
(mapa mapa-principal 1000 1000 ((objeto estabulo-vazio (80 80)) (objeto torre-mago (100 80)) (objeto castelo (100 100))))


In [8]:
(display "Localizando torre-mago no mapa-principal\n")
(display (encontra objeto 'torre-mago no mapa 'mapa-principal))

Localizando torre-mago no mapa-principal
(objeto torre-mago (100 80))

In [9]:
(display "Modificando nome do obj-1 para obj-modificado\n")
(muda 'nome do objeto 'estabulo-vazio no mapa 'mapa-principal para 'estabulo-cheio)
(display (encontra objeto 'estabulo-cheio no mapa 'mapa-principal))

Modificando nome do obj-1 para obj-modificado
(objeto estabulo-cheio (80 80))